### Hidden Recurrence vs Output Recurrence

To see how hidden and output recurrence work in practice, let's manually compute the forward pass for one of these recurrent types. Using the torch.nn module, a recurrent layer can be define via RNN, which is similar to the hidden-to-hidden recurrence. In the following code, we will create a recurrent layer from RNN and perform a forward pass on an input sequence of length 3 to compute the output. We will also manually compute the forward pass and compare the results with those of RNN.

First, let's create the layer and assign the weights and biases for our manual computations:

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(1)
rnn_layer = nn.RNN(input_size=5, hidden_size=2,
                   num_layers=1, batch_first=True)
w_xh = rnn_layer.weight_ih_l0
w_hh = rnn_layer.weight_hh_l0
b_xh = rnn_layer.bias_ih_l0
b_hh = rnn_layer.bias_hh_l0
print('W_xh shape:', w_xh.shape)
print('W_hh shape:', w_hh.shape)
print('b_xh shape:', b_xh.shape)
print('b_hh shape:', b_hh.shape)

W_xh shape: torch.Size([2, 5])
W_hh shape: torch.Size([2, 2])
b_xh shape: torch.Size([2])
b_hh shape: torch.Size([2])


The input shape for this layer is (batch_size, sequence_length, 5), where the first dimension is the batch dimension (as we set batch_first=True), the second dimension corresponds to the sequence, and the last dimension corresponds to the features. Notice that we will output a sequence, which, for an input sequence of length 3, will result in the output sequence $\langle o^{(0)}, o^{(1)}, o^{(2)} \rangle$. Also, RNN uses one layer by default, and you can set num_layers to stack multiple RNN layers together to form a stacked RNN. 

Now, we will call the forward pass on the rnn_layer and manually compute the outputs at each time step and compute them:

In [2]:
x_seq = torch.tensor([[1.0]*5, [2.0]*5, [3.0]*5]).float()
## output of the simple RNN:
output, hn = rnn_layer(torch.reshape(x_seq, (1, 3, 5)))
## manually computing the output:
out_man = []
for t in range(3):
    xt = torch.reshape(x_seq[t], (1, 5))
    print(f'Time step {t} =>')
    print('    Input                 :', xt.numpy())

    ht = torch.matmul(xt, torch.transpose(w_xh, 0, 1)) + b_xh
    print('    Hidden                :', ht.detach().numpy())

    if t > 0:
        prev_h = out_man[t-1]
    else:
        prev_h = torch.zeros((ht.shape))
    ot = ht + torch.matmul(prev_h, torch.transpose(w_hh, 0, 1)) + b_hh
    ot = torch.tanh(ot)
    out_man.append(ot)
    print('    Output (manual)       :', ot.detach().numpy())
    print('    RNN output            :', output[:, t].detach().numpy())
    print()

Time step 0 =>
    Input                 : [[1. 1. 1. 1. 1.]]
    Hidden                : [[-0.4701929  0.5863904]]
    Output (manual)       : [[-0.3519801   0.52525216]]
    RNN output            : [[-0.3519801   0.52525216]]

Time step 1 =>
    Input                 : [[2. 2. 2. 2. 2.]]
    Hidden                : [[-0.88883156  1.2364397 ]]
    Output (manual)       : [[-0.68424344  0.76074266]]
    RNN output            : [[-0.68424344  0.76074266]]

Time step 2 =>
    Input                 : [[3. 3. 3. 3. 3.]]
    Hidden                : [[-1.3074701  1.886489 ]]
    Output (manual)       : [[-0.8649416   0.90466356]]
    RNN output            : [[-0.8649416   0.90466356]]



In our manual forward computation, we used the hyperbolic tangent (tanh) activation function since it is also used in RNN (the default activation). As you can see from the printed results, the outputs from the manual forward computations exactly match the output of the RNN layer at each time step.

### The Challenges of Learning Long-Range Interactions

The structure of an LSTM cell and its underlying computations might seem very complex and hard to implement. However, the good news is that PyTorch has already implemented everything in optimized wrapper functions, which allows us to define our LSTM cells easily and efficiently. We will apply RNNs and LSTMs to real-world datasets in this chapter.

### Implementing RNNs for sequence modeling in PyTorch

Now that we have covered the underlying theory behind RNNs, we are ready to move on to the more practical portion of this chapter: implementing RNNs in PyTorch. During the rest of this chatper, we will apply RNNs to two common problem tasks:
- Sentiment analysis
- Language modeling
  
These two projects, which we will walk through together, are both fascinating but also quite involved. Thus, instead of providing the code all at once, we will break the implementatino up into several steps and discuss the code in detail. If you like to have a big picture overview and want to see all the code at once before diving into the discussion, take a look at the code implementation first.

#### Project One - Predicting the Sentiment of IMDb Movide Reviews

Sentiment analysis is concerned with analyzing the expressed opinion of a sentence or a text document. In this section, we will implement a multilayer RNN for sentiment analysis using a many-to-one architecture. 

In the next section, we will implement a many-to-many RNN for an application of language modeling. While the chosen examples are purposefully simple to introduce the main concepts of RNNs, language modeling has a wide range of interesting applications, such as building chatbots - giving computers the ability to directly talk and interact with humans.

##### Preparing the Movie Review Data

First, we will import the necessary modules and read the data from torchtext as follows:

In [3]:
import warnings
from tqdm import TqdmWarning


with warnings.catch_warnings():
    warnings.simplefilter("ignore", TqdmWarning)
    from huggingface_hub.utils import logging as hf_logging
    hf_logging.set_verbosity_error()
    from datasets import load_dataset
    imdb = load_dataset("stanfordnlp/imdb")

hf_logging.set_verbosity_warning()

train_dataset = imdb["train"]
test_dataset = imdb["test"]

Each set has 25,000 samples. And each sample of the dataset consists of two elements, the sentiment label representing the target label we want to predict (0 refers to negative sentiment and 1 refers to positive sentiment), and the movie review text(the input features). The text component of these movie reviews is sequences of words, and the RNN model classifies each sequence as a positive (1) or negative (0) review.

However, before we can feed the data into an RNN model, we need to apply several preprocessing steps:
- Split the training dataset into separate training and validation partitions
- Identify the unique words in the training dataset
- Map each unique word to a unique integer and encode the review text into encoded integers (an index of each unique word)
- Divide the dataset into mini-batches as input to the model

Let's proceed with the first step: creating a training and validation partition from the train_dataset we read earlier:

In [4]:
## Step 1: create the datasets
from torch.utils.data.dataset import random_split
torch.manual_seed(1)
train_dataset, valid_dataset = random_split(
    list(train_dataset), [20000, 5000])

The original training dataset contains 25,000 examples. 20,000 examples are randomly chosen for training, and 5,000 for validation.

To prepare data for input to an NN, we need to encode it into numeric values, as was mentioned in steps 2 and 3. To do this, we will first find the unique words (tokens) in the training dataset. While finding unique tokens is a process for which we can use Python datasets, it can be more efficient to use the Counter class from the collections package, which is part of Python's standard library.

In the following code, we will instantiate a new Counter object (token_counts) that will collect the unique word frequencies. Note that in this particular application (and in contrast to the bag-of-words model), we are only interested in the set of unique words and won't require the word counts, which are created as a side product. To split the text into words (or tokens), we will reuse the tokenizer function we developed in an earlier chapter, which also removes the HTML markups as well as punctuation and other non-letter characters.

The code for collecting unique tokens is as follows:

In [5]:
import re
from collections import Counter, OrderedDict

def tokenizer(text):
    text = re.sub(r'<[^>]*>', '', text)
    emoticons = re.findall(
        r'(?::|;|=)(?:-)?(?:\)|\(|D|P)', text.lower()
    )
    text = re.sub(r'[\W]+', ' ', text.lower()) + \
        ' '.join(emoticons).replace('-', '')
    tokenized = text.split()
    return tokenized

token_counts = Counter()
for item in train_dataset:
    line = item['text']
    label = item['label']
    tokens = tokenizer(line)
    token_counts.update(tokens)

print('Vocab-size:', len(token_counts))


Vocab-size: 69023


Next, we are going to map each unique word to a unique integer. This can be done manually using a Python dictionary, where the keys are the unique tokens (words) and the value associated with each key is a unique integer. We will also prepend two special tokens to the vocabulary - the padding and the unknown token:

In [6]:
sorted_tokens = [token for token, _ in token_counts.most_common()]
vocab = {"<pad>": 0, "<unk>": 1}
vocab.update({token: i for i, token in enumerate(sorted_tokens, start=2)})

To demonstrate the vocab dictionary, we will convert an example input text into a list of integer values:

In [7]:
print([vocab[token] for token in ['this', 'is', 'an', 'example']])

[11, 7, 35, 457]


Note that there might be some tokens in the validation or testing data that are not present in the training data and are thus not included in the mapping. If we have q tokens (that is, the size of token_counts passed to vocab, which in this case is 69,023), then all tokens that haven't been seen before, and are thus not included in token_counts, will be assigned integer 1 (a placeholder for the unknown token). In other words, the index 1 is reserved for unknown words. Another reserved value is the integer 0, which serves as a placeholder, a so-called padding token, for adjusting the sequence length. Later, when we are building an RNN model in PyTorch, we will consider this placeholder, 0, in more detail.

We can define the text_pipeline function to transform each text in the dataset accordingly:

In [8]:
## Step 3-A: Define encoding function
def text_pipeline(text):
    return [vocab.get(token, vocab["<unk>"]) for token in tokenizer(text)]

We will generate batches of samples using DataLoader and pass the text_pipeline:

In [9]:
## Step 3-B: Wrap the encode function
import numpy as np
from numpy.ma import ids


def collate_batch(batch):
    label_list = []
    text_list = []
    lengths = []
    for item in batch:
        label_list.append(item['label'])
        ids = np.asarray(text_pipeline(item["text"]), dtype=np.int64)
        processed_text = torch.from_numpy(ids)
        text_list.append(processed_text)
        lengths.append(processed_text.size(0))
    label_list = torch.tensor(label_list, dtype=torch.float32)
    lengths = torch.tensor(lengths)
    padded_text_list = nn.utils.rnn.pad_sequence(
        text_list, batch_first=True, padding_value=vocab['<pad>'])
    return padded_text_list, label_list, lengths

from torch.utils.data import DataLoader
dataloader = DataLoader(train_dataset, batch_size=4,
                        shuffle=False, collate_fn=collate_batch)


So far, we've converted sequences of words into sequences of integers. However, there is one issue that we need to resolve - the sequences currently have different lengths (as shown in the result of executing following code for four examples). Although, in general, RNNs can handle sequences with different lengths, we still need to make sure that all the sequences in a mini-batch have the same length to store them efficiently in a tensor.

PyTorch provides an efficient method, pad_sequence(), which will autoamtically pad the consecutive elements that are to be combined into a batch with placeholder values (0s) so that all sequences within a batch will have the same shape. In the previous code, we already created a data loader of a small batch_size from the training dataset and applied the collate_batch function, which itself included a pad_sequence() call.

However, to illustrate how padding works, we will take the first batch and print the sizes of the individual elements before combining these into mini-batches, as well as the dimensions of the resulting mini-batches:

In [10]:
text_batch, label_batch, length_batch = next(iter(dataloader))
print(text_batch)
print(label_batch)
print(length_batch)
print(text_batch.shape)

tensor([[   35,  1739,     7,   449,   721,     6,   301,     4,   787,     9,
             4,    18,    44,     2,  1705,  2460,   186,    25,     7,    24,
           100,  1874,  1739,    25,     7, 34415,  3568,  1103,  7517,   787,
             5,     2,  4991, 12401,    36,     7,   148,   111,   939,     6,
         11598,     2,   172,   135,    62,    25,  3199,  1602,     3,   928,
          1500,     9,     6,  4601,     2,   155,    36,    14,   274,     4,
         42945,     9,  4991,     3,    14, 10296,    34,  3568,     8,    51,
           148,    30,     2,    58,    16,    11,  1893,   125,     6,   420,
          1214,    27, 14542,   940,    11,     7,    29,   951,    18,    17,
         15994,   459,    34,  2480, 15211,  3713,     2,   840,  3200,     9,
          3568,    13,   107,     9,   175,    94,    25,    51, 10297,  1796,
            27,   712,    16,     2,   220,    17,     4,    54,   722,   238,
           395,     2,   787,    32,    27,  5236,  

As we can observe from the printed tensor shapes, the number of columns in the first batch is 218, which resulted from combining the first four examples into a single batch and using the maximum size of these examples. This means that the other three examples (whose lengths are 165, 86, and 145, respectively) in this batch are padded as much as necessary to match the size

Finally, let's divide all three datasets into data loaders with a batch size of 32:

In [11]:
batch_size = 32
train_dl = DataLoader(train_dataset, batch_size=batch_size,
                      shuffle=True, collate_fn=collate_batch)
valid_dl = DataLoader(valid_dataset, batch_size=batch_size,
                      shuffle=False, collate_fn=collate_batch)
test_dl = DataLoader(test_dataset, batch_size=batch_size,
                     shuffle=False, collate_fn=collate_batch)

Now, the data is in a suitable format for an RNN model, which we are going to implement in the following subsections. In the next subsection, however, we will first discuss feature embedding, which is an optional but highly recommended preprocessing step that is used to reduce the dimensionality of the word vectors.

##### Embedding Layers for Sentence Encoding

During the data preparation in the previous step, we generated sequences of the same length. The elements of these sequences were integer numbers that corresponded to the indices of unique words. These word indices can be converted into input features in several different ways. One naive way is to apply one-hot encoding to convert the indices into vectors of zeros and ones. Then, each word will be mapped to a vector whose size is the number of unique words in the entire dataset. Given that the number of unique words (the size of the vocabulary) can be in the order of $10^4 - 10^5$, which will also be the number of our input features, a model trained on such features may suffer from the curse of dimensionality. Furthermore, these features are very sparse since all are zero except one.

A more elegant approach is to map each word to a vector of a fixed size with real-valued elements (not necessarily integers). In contrast to the one-hot encoded vectors, we can use finite-sized vectors to represent an infinite number of real numbers.

This is the idea behind embedding, which is a feature-learning technique that we can utilize here to automatically learn the salient features to represent the words in our dataset. Given the number of unique words, $n_{words}$, we can select the size of the embedding vectors (a.k.a., embedding dimension) to be much smaller than the number of unique words to represent the entire vocabulary as input features.

The advantages of embedding over one-hot-encoding are as follows:
- A reduction in the dimensionality of the feature space to decrease the effect of the curse of dimensionality
- The extraction of salient features since the embedding layer in an NN can be optimized (or learned)

Given a set of tokens of size $n+2$ ($n$ is the size of the token set, plus index 0 is reserved for the padding placeholder, and 1 is for the words not present in the token set), an embedding matrix of size $(n+2) \times embedding\_dim$ will be created where each row of this matrix represents numeric features associated with a token. Therefore, when an integer index $i$ is given as input to the embedding, it will look up the corresponding row of the matrix at index $i$ and return numeric features. The embedding matrix serves as the input layer to our NN models. In practice, creating an embedding layer can simply be done using nn.Embedding. Let's see an example where we will create an embedding layer and apply it to a batch of two samples, as follows:

In [12]:
embedding = nn.Embedding(num_embeddings=10,
                         embedding_dim=3,
                         padding_idx=0)

# a batch of 2 samples of 4 indices each
text_encoded_input = torch.LongTensor([[1,2,4,5],[4,3,2,0]])
print(embedding(text_encoded_input))

tensor([[[ 0.7039, -0.8321, -0.4651],
         [-0.3203,  2.2408,  0.5566],
         [-0.4643,  0.3046,  0.7046],
         [-0.7106, -0.2959,  0.8356]],

        [[-0.4643,  0.3046,  0.7046],
         [ 0.0946, -0.3531,  0.9124],
         [-0.3203,  2.2408,  0.5566],
         [ 0.0000,  0.0000,  0.0000]]], grad_fn=<EmbeddingBackward0>)


The input to this model (embedding layer) must have rank 2 with the dimensionality $batchsize \times input\_length$, where $input\_length$ is the length of sequences (here, 4). For example, an input sequence in the minibatch could be $\langle 1, 5, 9, 2 \rangle$, where each element of this sequence is the index of the unique words. The output will have dimensionality $batchsize \times input\_length \times embedding\_dim$, where $embedding\_dim$ is the size of the embedding features (here, set to 3). The other argument provided to the embedding layer, $num\_embeddings$, corresponds to the unique integer values that the model will receive as input (for instance, $n+2$, set here to 10). Therefore, the embedding matrix in this case the size $10 \times 3$.

padding_idx indicates the token index for padding (here, 0), which, if specified, will not contribute to the gradient updates during training. In our example, the length of the original sequence of the second sample is 3, and we padded it with 1 more element 0. The embedding output of the padded element is [0, 0, 0].

##### Building an RNN Model

Now, we are ready to build an RNN model. Using the nn.Module class, we can combine the embedding layer, the recurrent layers of the RNN, and the fully connected non-recurrent layers. For the recurrent layers, we can use any of the following implementations:
- RNN: a regular RNN layer, that is, a fully connected recurrent layer
- LSTM: a long short-term memory RNN, which is useful for capturing the long-term dependencies
- GRU: a recurrent layer with a gated recurrent unit

To see how a multilayer RNN model can be built using one of these recurrent layres, in the following example, we will create an RNN model with two recurrent layers of type RNN. Finally, we will add a non-recurrent fully connected layer as the output layer, which will return a single output value as the prediction:

In [13]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, num_layers=2, 
                          batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)


    def forward(self, x):
        _, hidden = self.rnn(x)
        out = hidden[-1, :, :] # we use the final hidden state
                               # from the hidden layer as 
                               # the input to the fully connected
                               # layer
        
        out = self.fc(out)
        return out

model = RNN(64, 32)
print(model)
model(torch.randn(5, 3, 64))

RNN(
  (rnn): RNN(64, 32, num_layers=2, batch_first=True)
  (fc): Linear(in_features=32, out_features=1, bias=True)
)


tensor([[ 0.3183],
        [ 0.1230],
        [ 0.1772],
        [-0.1052],
        [-0.1259]], grad_fn=<AddmmBackward0>)

As you can see, building an RNN model using these recurrent layers is pretty straightforward. In the next subsection, we will go back to our sentiment analysis task and build an RNN model to solve that.

##### Building an RNN Model for the Sentiment Analysis Task

Since we have very long sequences, we are going to use an LSTM layer to account for long-range effects. We will create an RNN model for sentiment analysis, starting with an embedding layer producing word embeddings of feature size 20 (embed_dim = 20). Then, a recurrent layer of type LSTM will be added. Finally, we will add a fully connected layer as a hidden layer and another fully connected layer as the output layer, which will return a single class-membership probability value via the logistic sigmoid activation as the prediction:

In [14]:
class RNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, rnn_hidden_size, 
                 fc_hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,
                                      embed_dim,
                                      padding_idx=0)
        self.rnn = nn.LSTM(embed_dim, rnn_hidden_size,
                           batch_first=True)
        self.fc1 = nn.Linear(rnn_hidden_size, fc_hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(fc_hidden_size, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, text, lengths):
        out = self.embedding(text)
        out = nn.utils.rnn.pack_padded_sequence(
            out, lengths.cpu().numpy(), enforce_sorted=False, batch_first=True
        )
        out, (hidden, cell) = self.rnn(out)
        out = hidden[-1, :, :]
        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.sigmoid(out)
        return out

vocab_size = len(vocab)
embed_dim = 20
rnn_hidden_size = 64
fc_hidden_size = 64
torch.manual_seed(1)
model = RNN(vocab_size, embed_dim, rnn_hidden_size, fc_hidden_size)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model

RNN(
  (embedding): Embedding(69025, 20, padding_idx=0)
  (rnn): LSTM(20, 64, batch_first=True)
  (fc1): Linear(in_features=64, out_features=64, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

Now we will develop the train function to train the model on the given dataset for one epoch and return the classification accuracy and loss:

In [15]:
def train(dataloader):
    model.train()
    total_acc, total_loss = 0, 0
    for text_batch, label_batch, lengths in dataloader:
        # Move the batches to the appropriate device
        text_batch = text_batch.to(device)
        label_batch = label_batch.to(device)
        
        optimizer.zero_grad()
        pred = model(text_batch, lengths)[:, 0]
        loss = loss_fn(pred, label_batch)
        loss.backward()
        optimizer.step()
        total_acc += (
            (pred >= 0.5).float() == label_batch
        ).float().sum().item()
        total_loss += loss.item()*label_batch.size(0)
    return total_acc/len(dataloader.dataset), \
           total_loss/len(dataloader.dataset)

Similarly, we will develop the evaluate function to measure the model's performance on a given dataset:

In [16]:
def evaluate(dataloader):
    model.eval()
    total_acc, total_loss = 0, 0
    with torch.no_grad():
        for text_batch, label_batch, lengths in dataloader:
            # Move the batches to the appropriate device
            text_batch = text_batch.to(device)
            label_batch = label_batch.to(device)
            
            pred = model(text_batch, lengths)[:, 0]
            loss = loss_fn(pred, label_batch)
            total_acc += (
                (pred >= 0.5).float() == label_batch
            ).float().sum().item()
            total_loss += loss.item()*label_batch.size(0)
    return total_acc/len(dataloader.dataset), \
           total_loss/len(dataloader.dataset)

The next step is to create a loss function and optimizer (Adam optimizer). For a binary classification with a single class-membership probability output, we use the binary cross-entropy loss (BCELoss) as the loss function:

In [17]:
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

Now, we will train the model for 10 epochs and display the training and validation performances:

In [18]:
num_epochs = 10
torch.manual_seed(1)
for epoch in range(num_epochs):
    acc_train, loss_train = train(train_dl)
    acc_valid, loss_valid = evaluate(valid_dl)
    print(f'Epoch {epoch} accuracy: {acc_train:.4f}'
          f' val_accuracy: {acc_valid:.4f}')

Epoch 0 accuracy: 0.5772 val_accuracy: 0.6326
Epoch 1 accuracy: 0.7228 val_accuracy: 0.7686
Epoch 2 accuracy: 0.8331 val_accuracy: 0.8260
Epoch 3 accuracy: 0.8855 val_accuracy: 0.8464
Epoch 4 accuracy: 0.9163 val_accuracy: 0.8588
Epoch 5 accuracy: 0.9395 val_accuracy: 0.8654
Epoch 6 accuracy: 0.9555 val_accuracy: 0.8426
Epoch 7 accuracy: 0.9699 val_accuracy: 0.8480
Epoch 8 accuracy: 0.9791 val_accuracy: 0.8698
Epoch 9 accuracy: 0.9860 val_accuracy: 0.8688


After training the model for 10 epochs, we will evaluate it on the test dataset:

In [19]:
acc_test, _ = evaluate(test_dl)
print(f'test_accuracy: {acc_test:.4f}')

test_accuracy: 0.8615


It showed 86 percent accuracy. (Note that this result is not the best when compared to the state-of-the-art methods used on the IMDb dataset. The goal was simply to show how an RNN works in PyTorch.)

##### More On the Bidirectional RNN

In addition, we will set the bidirectional configuration of the LSTM to True, which will make the recurrent layer pass through the input sentences from both directions, start to end, as well as in the reverse direction:

In [20]:
class RNN(nn.Module):
    def __init__(self, vocab_size, embed_dim,
                 rnn_hidden_size, fc_hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(
            vocab_size, embed_dim, padding_idx=0
        )
        self.rnn = nn.LSTM(embed_dim, rnn_hidden_size,
                           batch_first=True, bidirectional=True)
        self.fc1 = nn.Linear(rnn_hidden_size*2, fc_hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(fc_hidden_size, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, text, lengths):
        out = self.embedding(text)
        out = nn.utils.rnn.pack_padded_sequence(
            out, lengths.cpu().numpy(), enforce_sorted=False, batch_first=True
        )
        _, (hidden, cell) = self.rnn(out)
        out = torch.cat((hidden[-2, :, :],
                         hidden[-1, :, :]), dim=1)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.sigmoid(out)
        return out

torch.manual_seed(1)
model = RNN(vocab_size, embed_dim,
            rnn_hidden_size, fc_hidden_size)
model

RNN(
  (embedding): Embedding(69025, 20, padding_idx=0)
  (rnn): LSTM(20, 64, batch_first=True, bidirectional=True)
  (fc1): Linear(in_features=128, out_features=64, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

The bidirectional RNN layer makes two passes over each input sequence: a forward pass and a reverse or backward pass (note that this is not to be confused with the forward and backward passes in the context of backpropagation). The resulting hidden states of these forward and backward passes are usually concatenated into a single hidden state. Other merge modes include summation, multiplication (multiplying the results of the two passes), and averaging (taking the average of the two).

We can also try other types of recurrent layers, such as the regular RNN. However, as it turns out, a model built with regular recurrent layers won't be able to reach a good predictive performance (even on the training data). For example, if you try replacing the bidirectional LSTM layer in the previous code with a unidirectional nn.RNN (instead of nn.LSTM) layer and train the model on full-length sequences, you may observe that the loss will not even decrease during training. The reason is that the sequences in this dataset are too long, so a model with an RNN layer cannot learn the long-term dependencies and may suffer from vanishing or exploding gradient problems.

#### Project Two - Character-Level Language Modeling in PyTorch

Language modeling is a fascinating application that enables machines to perform human language related tasks, such as generating English sentences. 

In the model that we will build now, the input is a text document, and our goal is to develop a model that can generate new text that is similar in style to the input document. Examples of such input are a book or a computer program in a specific programming language.

In character-level language modeling, the input is broken down into a sequence of characters that are fed into our network one character at a time. The network will process each new character in conjunction with the memory of the previously seen charaters to predict the next one.

##### Preprocessing the Dataset

In this section, we will prepare the data for character-level language modeling.

To obtain the input data, visit the Project Gutenberg website at https://www.gutenberg.org/, which provides thousands of free e-books.

Note that this link will take you directly to the download page. If you are using macOS or a Linux operating system, you can download the file with the following command in the terminal:

Once we have downloaded the dataset, we can read it into a Python session as plain text. Using the following code, we will read the text directly from the downloaded file and remove portions from the beginning and the end (these contain certain descriptions of the Gutenberg project). Then, we will create a Python variable, char_set, that represents the set of unique characters observed in this text:

In [21]:
import numpy as np
## Reading and processing text
with open('1268-0.txt', 'r', encoding='utf8') as fp:
    text = fp.read()
start_indx = text.find('THE MYSTERIOUS ISLAND')
end_indx = text.find('*** END OF THE PROJECT GUTENBERG EBOOK 1268 ***')
text = text[start_indx:end_indx]
char_set = set(text)
print('Total Length:', len(text))
print('Unique Characters:', len(char_set))

Total Length: 1112263
Unique Characters: 79


After downloading and preprocessing the text, we have a sequence consisting of 1,112,263 characters in total and 79 unique characters. However, most NN libraries and RNN implementations cannot deal with input data in string format, which is why we have to convert the text into a numeric format. To do this, we will create a simple Python dictionary that maps each character to an integer, char2int. We will also need a reverse mapping to convert the results of our model back to text. Although the reverse can be done using a dictionary that associates integer keys with character vaues, using a NumPy array and indexing the array to map indices to those unique characters is more efficient. 

Building the dictionary to map characters to integers, and reverse mapping via indexing a NumPy array is as follows:

In [22]:
chars_sorted = sorted(char_set)
char2int = {ch: i for i, ch in enumerate(chars_sorted)}
char_array = np.array(chars_sorted)
text_encoded = np.array(
    [char2int[ch] for ch in text],
    dtype=np.int32
)
print('Text encoded shape:', text_encoded.shape)
print(text[:15], '== Encoding ==>', text_encoded[:15])
print(text_encoded[15:21], '== Reverse ==>',
      ''.join(char_array[text_encoded[15:21]]))

Text encoded shape: (1112263,)
THE MYSTERIOUS  == Encoding ==> [43 31 28  1 36 47 42 43 28 41 32 38 44 42  1]
[32 42 35 24 37 27] == Reverse ==> ISLAND


The text_encoded NumPy array contains the encoded values for all the characters in the text. Now, we will print out the mappings of the first five characters from this array:

In [23]:
for ex in text_encoded[:5]:
    print('{} -> {}'.format(ex, char_array[ex]))

43 -> T
31 -> H
28 -> E
1 ->  
36 -> M


Now, let's step back and look at the big picture of what we are trying to do. For the text generation task, we can formulate the problem as a classification task.

Suppose we have a set of sequences of text characters that are incomplete. In order to generate new text, our goal is to design a model that can predict the next character of a given input sequence, where the input sequence represents an incomplete text. For example, after seeing "Deep Learn," the model should predict "i" as the next character. Given that we have 79 unique characters, this problem becomes a multiclass classification task.

Starting with a sequence of length 1 (that is, one single letter), we can iteratively generate new text based on this multiclass classification approach.

To implement the text generation task in PyTorch, let's first clip the sequence length to 40. This means that the input tensor, $x$, consists of 40 tokens. In practice, the sequence length impacts the quality of the generated text. Longer sequences can result in more meaningful sentences. For shorter sequences, however, the model might focus on capturing individual words correctly, while ignoring the context for the most part. Although longer sequences usually result in more meaningful sentences, as mentioned, for long sequences, the RNN model will have problems capturing long-range dependencies. Thus, in practice, finding a sweet spot and good value for the sequence length is a hyperparameter optimization problem, which we have to evaluate empirically. Here, we are going to choose 40, as it offers a good trade-off.

The inputs $x$, and targets $y$, are offset by one character. Hence, we will split the text into chunks of size 41: the first 40 characters will form the input sequence, $x$, and the last 40 elements will for the target sequence, $y$.

We have already stored the entire encoded text in its original order in text_encoded. We will first create text chunks consisting of 41 characters each. We will further get rid of the last chunk if it is shorter than 41 characters. As a result, the new chunked dataset, named text_chunks, will always contain sequences of size 41. The 41-character chunks will then be used to construct the sequence $x$ (that is, the input), as well as the sequence $y$ (that is, the target), both of which will have 40 elements. For instance, sequence $x$ will consist of the elements with indices $[0, 1, \cdots, 39]$. Furthermore, since sequence $y$ will be shifted by one position with respect to $x$, its corresponding indices will be $[1, 2, \cdots, 40]$. Then, we will transform the result into a Dataset object by applying a self-defined Dataset class:

In [24]:
import torch
from torch.utils.data import Dataset
seq_length = 40
chunk_size = seq_length + 1
text_chunks = [text_encoded[i:i+chunk_size]
               for i in range(len(text_encoded) - chunk_size + 1)]

class TextDataset(Dataset):

    def __init__(self, text_chunks):
        self.text_chunks = text_chunks

    def __len__(self):
        return len(self.text_chunks)

    def __getitem__(self, idx):
        text_chunk = self.text_chunks[idx]
        return text_chunk[:-1].long(), text_chunk[1:].long()

chunks_array = np.asarray(text_chunks)
chunks_tensor = torch.from_numpy(chunks_array).long()
seq_dataset = TextDataset(chunks_tensor)
for i, (seq, target) in enumerate(seq_dataset):
    print(' Input (x): ', 
          repr(''.join(char_array[seq])))
    print('Target (y): ',
          repr(''.join(char_array[target])))
    print()
    if i == 1:
        break

 Input (x):  'THE MYSTERIOUS ISLAND\n\nby Jules Verne\n\n1'
Target (y):  'HE MYSTERIOUS ISLAND\n\nby Jules Verne\n\n18'

 Input (x):  'HE MYSTERIOUS ISLAND\n\nby Jules Verne\n\n18'
Target (y):  'E MYSTERIOUS ISLAND\n\nby Jules Verne\n\n187'



Finally, the last step in perparing the dataset is to transform this dataset into mini-batches:

In [25]:
from torch.utils.data import DataLoader
batch_size = 64
torch.manual_seed(1)
seq_dl = DataLoader(seq_dataset, batch_size=batch_size,
                    shuffle=True, drop_last=True)

##### Building a Character-Level RNN Model

Now that the dataset is ready, building the model will be relatively straightforward:

In [26]:
import torch.nn as nn
class RNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, rnn_hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn_hidden_size = rnn_hidden_size
        self.rnn = nn.LSTM(embed_dim, rnn_hidden_size,
                          batch_first=True)
        self.fc = nn.Linear(rnn_hidden_size, vocab_size)

    def forward(self, x, hidden, cell):
        out = self.embedding(x).unsqueeze(1)
        out, (hidden, cell) = self.rnn(out, (hidden, cell))
        out = self.fc(out).reshape(out.size(0), -1)
        return out, hidden, cell

    def init_hidden(self, batch_size):
        hidden = torch.zeros(1, batch_size, self.rnn_hidden_size)
        cell = torch.zeros(1, batch_size, self.rnn_hidden_size)
        return hidden, cell


Notice that we will need to have the logits as outputs of the model so that we can sample from the model predictions in order to generate new text. We will get to this sampling part:

Then we can specify the model parameters and create an RNN model:

In [27]:
vocab_size = len(char_array)
embed_dim = 256
rnn_hidden_size = 512
torch.manual_seed(1)
model = RNN(vocab_size, embed_dim, rnn_hidden_size)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model


RNN(
  (embedding): Embedding(79, 256)
  (rnn): LSTM(256, 512, batch_first=True)
  (fc): Linear(in_features=512, out_features=79, bias=True)
)

The next step is to create a loss function and optimizer (Adam optimizer). For a multiclass classification (we have vocab_size = 79 classes) with a single logits output for each target character, we use CrossEntropyLoss as the loss function:

In [28]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

Now we will train the model for 10,000 epochs. In each epoch, we will use only one batch randomly chosen from the data loader, seq_dl. We will also display the training loss for every 500 epochs:

In [29]:
num_epochs = 10000
torch.manual_seed(1)
for epoch in range(num_epochs):
    seq_batch, target_batch = next(iter(seq_dl))
    seq_batch = seq_batch.to(device)
    target_batch = target_batch.to(device)
    hidden, cell = model.init_hidden(seq_batch.size(0))
    hidden, cell = hidden.to(device), cell.to(device)  
    optimizer.zero_grad()
    loss = 0
    for c in range(seq_length):
        pred, hidden, cell = model(seq_batch[:, c], hidden, cell)
        loss += loss_fn(pred, target_batch[:, c])
    loss.backward()
    optimizer.step()
    loss = loss.item()/seq_length
    if epoch % 500 == 0:
        print(f'Epoch {epoch} loss: {loss:.4f}')

Epoch 0 loss: 4.3878
Epoch 500 loss: 1.4512
Epoch 1000 loss: 1.2810
Epoch 1500 loss: 1.2307
Epoch 2000 loss: 1.2042
Epoch 2500 loss: 1.2267
Epoch 3000 loss: 1.1332
Epoch 3500 loss: 1.2111
Epoch 4000 loss: 1.1985
Epoch 4500 loss: 1.2029
Epoch 5000 loss: 1.1961
Epoch 5500 loss: 1.0567
Epoch 6000 loss: 1.1270
Epoch 6500 loss: 1.1162
Epoch 7000 loss: 1.1779
Epoch 7500 loss: 1.1444
Epoch 8000 loss: 1.1877
Epoch 8500 loss: 1.1300
Epoch 9000 loss: 1.1892
Epoch 9500 loss: 1.1724


Next, we can evaluate the model to generate new text, starting with a given short string. In the next section, we will define a function to evaluate the trained model.

##### Evaluation Phase - Generating New Text Passages

The RNN model we trained in the previous section returns the logits of size 79 for each unique character. These logits can be readily converted to probabilities, via the softmax function, that a particular character will be encountered as the next character. To predict the next character in the sequence, we can simply select the element with the maximum logit value, which is equivalent to selecting the character with the highest probability. However, instead of always selecting the character with the highest likelihood, we want to (randomly) sample from the outputs; otherwise the model will always produce the same text. PyTorch already provides a class, torch.distributions.categorical.Categorical, which we can use to draw random samples from a categorical distribution. To see how this works, let's generate some random samples from three categories [0, 1, 2], with input logits [1, 1, 1]:

In [30]:
from torch.distributions.categorical import Categorical
torch.manual_seed(1)
logits = torch.tensor([[1.0, 1.0, 1.0]])
print('Probabilities:', nn.functional.softmax(logits, dim=1).numpy()[0])
m = Categorical(logits=logits)
samples = m.sample((10,))
print(samples.numpy())

Probabilities: [0.33333334 0.33333334 0.33333334]
[[0]
 [0]
 [0]
 [0]
 [1]
 [0]
 [1]
 [2]
 [1]
 [1]]


As you can see, with the given logits, the categories have the same probabilities. Therefore, if we use a large sample size, we would expect the number of occurrences of each category to reach $\approx$ 1/3 of the sample size. If we change the logits to [1, 1, 3], then we would expect to observe more occurences for category 2 (when a very large number of examples are drawn from this distribution):

In [31]:
torch.manual_seed(1)
logits = torch.tensor([[1.0, 1.0, 3.0]])
print('Probabilities:', nn.functional.softmax(logits, dim=1).numpy()[0])
m = Categorical(logits=logits)
samples = m.sample((10,))
print(samples.numpy())

Probabilities: [0.10650698 0.10650698 0.78698605]
[[0]
 [2]
 [2]
 [1]
 [2]
 [1]
 [2]
 [2]
 [2]
 [2]]


Using Categorical, we can generate examples based on the logits computed by our model.

We will define a function, sample(), that receives a short starting string, starting_str, and generate a new string, generated_str, which is initially set to the input string. starting_str is encoded to a sequence of integers, encoded_input. encoded_input is passed to the RNN model one character at a time to update the hidden states. The last character of encoded_input is passed to the model to generate a new character. Note that the output of the RNN model represents the logits (here, a vector of size 79, which is the total number of possible characters) for the next character after observing the input sequence by the model.

Here, we only use the logits output (that is, $o^{(T)}$), which is passed to the Categorical class to generate a new sample. This new sample is converted to a character, which is then appended to the end of the generated string, generated_text, increasing its length by 1. Then, this process is repeated until the length of the generated string reaches the desired value. The processs of consuming the generated sequence as input for generating new elements is called **autoregression**.

The code for the sample() function is as follows:


In [32]:
@torch.no_grad()
def sample(model, starting_str, 
           len_generated_text=500, 
           scale_factor=1.0):
    encoded_input = torch.tensor([char2int[ch] for ch in starting_str],
                                 dtype=torch.long,
                                 device=device).view(1, -1)
    generated_str = starting_str

    model.eval()
    hidden, cell = model.init_hidden(1)
    hidden, cell = hidden.to(device), cell.to(device)
    for c in range(len(starting_str) - 1):
        _, hidden, cell = model(
            encoded_input[:, c].view(1), hidden, cell
        )
    last_char = encoded_input[:, -1]
    for i in range(len_generated_text):
        logits, hidden, cell = model(
            last_char.view(1), hidden, cell
        )
        logits = torch.squeeze(logits, 0)
        scaled_logits = logits * scale_factor
        m = Categorical(logits=scaled_logits)
        last_char = m.sample()
        generated_str += str(char_array[last_char.item()])

    return generated_str

Now, let's generate some new text:

In [33]:
torch.manual_seed(1)
print(sample(model, starting_str='The island'))


The islands of concured, thanks to there.

“To! then,” repeated the reporter?

Now, Pencroft could do it with which they rereads these water, and for which Top, and the sailor had not attempt, and of the pole appeared, when these sock were in torn a vast, whose ponding substance the beach side. He had also that he had eneagered get against Jacamar Wood, and I recountry the means
which gave the bright service, looked unamong voice.

Thus, even whones you, and these? But there was not suspuration after havi


As we can see, the model generates mostly correct words, and, in some cases, the sentences are partially meaningful. We can further tune the training parameters, such as the length of the input sequences for training, and the model architecture.

Furthermore, to control the predictability of the generated samples (that is, generating text following the learned patterns from the training text versus adding more randomness), the logits computed by the RNN model can be scaled before being passed to Categorical for sampling. The scaling factor, $\alpha$ can be interpreted as an analog to the temperature in physics. Higher temperature results in more entropy or randomness versus more predictable behavior at lower temperatures. By scaling the logits with $\alpha < 1$, the probabilities computed by the softmax function become more uniform, as shown in the following code:

In [35]:
logits = torch.tensor([[1.0, 1.0, 3.0]])
print('Probabilities before scaling:        ',
      nn.functional.softmax(logits, dim=1).numpy()[0])
print('Probabilities after scaling with 0.5:',
      nn.functional.softmax(logits * 0.5, dim=1).numpy()[0])
print('Probabilities after scaling with 0.1:',
      nn.functional.softmax(logits * 0.1, dim=1).numpy()[0])

Probabilities before scaling:         [0.10650698 0.10650698 0.78698605]
Probabilities after scaling with 0.5: [0.21194156 0.21194156 0.57611686]
Probabilities after scaling with 0.1: [0.3104238  0.3104238  0.37915248]


As we can see, scaling the logits by $\alpha = 1$ results in near-uniform probabilities $[0.31, 0.31, 0.38]$. Now, we can compare the generated text with $\alpha = 2.0$ and $\alpha = 0.5$, as shown in the following points:

In [36]:
torch.manual_seed(1)
print(sample(model, starting_str='The island', scale_factor=2.0))

The island was concealed the rocks, and then, the poor boy, they walking in the boat. The settlers did not save him of the rocks.”

“Will that the oxygency to me the mast of the channel and it,” replied Pencroft, “that will be doubted to the sea, it was there is not for a suppose that they were easy,” replied Pencroft, “that it was agreed to come from the colonists, who was evidently again the sea, which was not to see the
convicts were deposited they could not say nothing behind the sea, and that he had 


In [37]:
torch.manual_seed(1)
print(sample(model, starting_str='The island', scale_factor=0.5))

The islands offerect with asinky, it reason,” returned Cyluenched a tillaily?-yarkast gives eceregize it.”
-avidfe? Gidearsh,”!””” ,soinHes can you?” Hopiced does also pusched his wriloty, wisned woftle
polgeasades initoly dibts, it quarked.

No, in 5
home there Neb fove this Leeling?”
 “Anot!
Meangy? Mr.
So,lish, byeke eagarly.”

“SUppons were den in?” nourished own Rotariolatic ween, a Smoke by nimile’-f,--not  we coumy?
Nvoittently-.-”

“If hon-ixyletowedtived,--fork unvin-tid.”

Mysta alith affrighing


The results show that scaling the logits with $\alpha = 0.5$ (increasing the temperature) generates more random text. There is a trade-off between the novelty of the generated text and its correctness.

In this section, we worked with character-level text generation, which is a sequence-to-sequence (seq2seq) modeling task. While this example may not be very useful by itself, it is easy to think of several useful applications for these types of models; for example, a similar RNN model can be trained as a chatbot to assist users with simple queries.